# CityEye — Colab Training
Run on T4 GPU: Runtime → Change runtime type → T4 GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/cityeye'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/videos', exist_ok=True)

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
!git clone https://github.com/vashika2/cityeye.git
%cd cityeye
!pip install -q -r requirements.txt
!pip install -q lightning albumentations wandb onnx onnxruntime

In [ ]:
import os, shutil
os.makedirs('data', exist_ok=True)
video_dir = f'{DRIVE_DIR}/videos'
for v in os.listdir(video_dir):
    if v.endswith('.mp4'):
        shutil.copy(f'{video_dir}/{v}', f'data/{v}')
        print(f'Copied: {v}')

In [ ]:
!python scripts/prepare_dataset.py --video_dir data --output_dir data/processed --fps 5 --max_frames 500

In [ ]:
import os
os.environ['WANDB_MODE'] = 'disabled'
!python scripts/train.py --config configs/default.yaml

In [ ]:
import shutil, glob
checkpoints = glob.glob('runs/train/cityeye-best-acc*/*.ckpt')
if checkpoints:
    shutil.copy(checkpoints[0], f'{DRIVE_DIR}/checkpoints/best_model.ckpt')
    print(f'Saved: {checkpoints[0]}')
else:
    shutil.copy('runs/train/last.ckpt', f'{DRIVE_DIR}/checkpoints/last.ckpt')
    print('Saved last checkpoint')

In [ ]:
import glob
ckpt = glob.glob('runs/train/cityeye-best-acc*/*.ckpt')
ckpt = ckpt[0] if ckpt else 'runs/train/last.ckpt'
!python scripts/evaluate.py --config configs/default.yaml --checkpoint "{ckpt}"